# Tutorial: single-trial E/I population data

The `results/trial_counts_b<N>_<animal>_s<session>.npz` files hold **single-trial spike
counts** of the E (wide-spike) and I (narrow-spike) populations. Time is binned (default
1 ms) and each bin holds the *number* of spikes in it. There is no trial averaging,
smoothing or normalisation. This notebook shows how to

1. choose the trial window (before generating files),
2. load a file and select trials by condition,
3. trim the time axis and rebin,
4. look at single trials,
5. compute trial averages and the paper's baseline-normalised PSTH,
6. pool across mice, and make cross-validation folds.

Field reference: `DATA.md`. Files are written by `regenerate_population_rates.py`.
The reducing helpers used here (`select`, `rebin`, `psth`, `fold_labels`) live in
`trial_analysis.py`; each is a few lines, so read them there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import data_loader
import trial_analysis as ta

ANIMAL = "M150605_ICTP1"

## 1. Choosing the trial window

Trials are not stored individually anywhere in the raw data. Each experiment is one
continuous recording with hundreds of trials back to back. We find each trial's laser
onset and cut a window `[-pre_s, +post_s)` around it (`PRE_S` / `POST_S` in
`regenerate_population_rates.py`). A trial whose window runs past the end of its
experiment's recording is **dropped** (in practice an experiment's last trial or two). A window longer than the gap to the next trial runs into the next trial.

`data_loader.get_trial_margins` reports, for every laser trial, how much recording there
is around its onset. It reads only trial timing (the `.ns5` laser channels), not spikes,
and needs the raw data mounted at `/mnt/scratch` (~2–3 min per mouse).

In [ ]:
margins = data_loader.get_trial_margins(base_dir=f"/mnt/scratch/{ANIMAL}", animal_id=ANIMAL)

fig, axs = plt.subplots(1, 3, figsize=(13, 3))
for ax, key in zip(axs, ["pre_avail_s", "post_avail_s", "next_gap_s"]):
    v = margins[key][np.isfinite(margins[key])]
    ax.hist(np.minimum(v, 5), bins=50)    # long tail piled into the 5 s bin
    ax.set_yscale("log")                  # the few short margins are what matter
    ax.set_xlabel(f"{key} (clipped at 5 s)"); ax.set_ylabel("trials")
fig.tight_layout()

`window_trial_loss` turns that into a cost for a candidate window.
- `n_dropped` trials would be excluded.
- `n_next_trial_in_window` / `n_prev_trial_in_window` trials are kept but overlap a
  neighbouring trial. Whether to exclude them downstream is your call.

In [ ]:
for pre_s, post_s in [(0.5, 1.5), (0.5, 3.0), (1.0, 1.5), (2.0, 4.0)]:
    print(f"[-{pre_s}, +{post_s}) s:", data_loader.window_trial_loss(margins, pre_s, post_s))

`trial_spacing_summary` compares, per experiment type, the window a trial needs
(`pre_s` before the first pulse to `post_last_s` after the end of the last pulse) with
the shortest and median onset-to-onset gap to the next trial:

In [ ]:
for t, r in data_loader.trial_spacing_summary(margins, pre_s=1.0, post_last_s=0.2).items():
    print(f"{t:10s} need {r['required_len_s']:.2f} s   min gap {r['min_gap_s']:.2f} s   "
          f"median gap {r['median_gap_s']:.2f} s   n={r['n_trials']}")

## 2. Loading a file and selecting trials

The file is two tables of parallel arrays:

- **Trial table** (axis 0 = trial): `counts (n_trials, 2, n_bins)`, `cond_idx`, `exp_num`, `trial_in_exp`, `onset_sample`.
- **Condition table** (axis 0 = condition): `cond_exp_type`, `cond_ipi_ms`, `cond_dur1_ms`, and the other `cond_*` arrays.

`cond_idx[i]` is the condition row of trial `i`. `counts` axis 1 is `0 = E`, `1 = I`.

Two condition columns matter for almost every analysis (README "Protocol parameters"):
- `cond_contrast`: some experiments fire the laser *during a screen stimulus*
  (`contrast = 100`; README "RandNoise"). The paper's analyses use a blank screen, so
  select `contrast=0` unless you want those trials.
- `cond_ipi_ms` is Protocol `intT`, the gap from the **end** of pulse 1 to the **start**
  of pulse 2. `cond_onset_ipi_ms` (= intT + dur1) is onset-to-onset.

The same laser-only stimulus can appear as several conditions, because experiments differ
in `Tp` (NaN for protocols without it). Pool them with `ta.select`, which ignores
columns you don't name.

In [ ]:
d = ta.load(ANIMAL)                       # = dict(np.load("results/trial_counts_b30_<animal>_s1.npz"))
t = d["time_axis"]
bin_s = ta.bin_s(d)

print("counts", d["counts"].shape, d["counts"].dtype, "| bin", bin_s * 1e3, "ms",
      "| window", float(d["pre_s"]), float(d["post_s"]), "| dropped", int(d["n_trials_dropped"]))

The condition table, with each condition's trial count:

In [ ]:
n_per_cond = np.bincount(d["cond_idx"], minlength=d["cond_exp_type"].size)
for k in range(d["cond_exp_type"].size):
    print(f"{k:3d}  {d['cond_exp_type'][k]:10s} ipi={d['cond_ipi_ms'][k]:5d} ms  "
          f"dur={d['cond_dur1_ms'][k]:g}/{d['cond_dur2_ms'][k]:g} ms  "
          f"contrast={d['cond_contrast'][k]:3d}  Tp={d['cond_tp_ms'][k]:g}  n={n_per_cond[k]}")

To select trials, build a boolean mask on the condition table and map it to trials
through `cond_idx`. `ta.select` does this for any combination of condition columns:

In [ ]:
m = ta.select(d, exp_type="single_E")
print("single_E trials:", m.sum(), "in conditions", np.unique(d["cond_idx"][m]))
print("  of which laser-only (contrast 0):", ta.select(d, exp_type="single_E", contrast=0).sum())
print("paired_EE at ipi 40 ms, laser-only:", ta.select(d, exp_type="paired_EE", ipi_ms=40, contrast=0).sum())

## 3. Trimming and rebinning

Trimming is slicing the last axis. Rebinning sums adjacent bins. That is exact because
these are counts, so the files keep 1 ms and you coarsen as needed.

In [ ]:
keep = (t >= -0.2) & (t < 0.5)
x_trim, t_trim = d["counts"][..., keep], t[keep]

c10, t10 = ta.rebin(d["counts"], t, 10)   # 1 ms -> 10 ms
print(x_trim.shape, c10.shape)

## 4. Single trials

Each row of `counts` is one trial. At 1 ms most bins hold 0 or 1 spikes, so a single
trial is easiest to read as a raster of spike counts. The heat map stacks every trial of
one condition (in recording order), and the bottom panels show three individual trials
at 10 ms. The post-pulse silence (~20–150 ms) and the rebound show in nearly every
trial. The brief evoked burst at onset is sparse on single trials, which is one reason
the paper works with trial averages.

In [ ]:
trials = np.flatnonzero(ta.select(d, exp_type="single_E", dur1_ms=1.0, contrast=0))
win = (t >= -0.2) & (t < 0.6)
win10 = (t10 >= -0.2) & (t10 < 0.6)

fig, axs = plt.subplots(2, 2, figsize=(12, 6), sharex=True, height_ratios=[2, 1])
for pop, name in enumerate("EI"):
    ax = axs[0, pop]
    ax.imshow(d["counts"][trials][:, pop, win], aspect="auto", interpolation="nearest",
              cmap="Greys", vmax=2, extent=[-0.2, 0.6, len(trials), 0])
    ax.set_title(f"{name} spikes per 1 ms bin — single_E, 1 ms, laser only, {len(trials)} trials")
    ax.set_ylabel("trial")
    ax = axs[1, pop]
    for i in trials[:3]:
        ax.step(t10[win10], c10[i, pop, win10] / 0.01, where="mid",
                label=f"exp {d['exp_num'][i]}, trial {d['trial_in_exp'][i]}")
    ax.set_xlabel("time from onset (s)"); ax.set_ylabel("Hz (10 ms bins)")
axs[1, 0].legend(fontsize=7)
for ax in axs.ravel():
    ax.axvline(0, color="tab:blue", lw=0.8)
fig.tight_layout()

## 5. Averages and the paper's PSTH

The paper's population activity is the **trial-averaged** rate divided by its own mean
over the baseline window (−0.5 to −0.1 s), smoothed with a 40 ms Hamming window.
`ta.psth` averages first and then normalises. Dividing each trial by its own baseline is
unstable (single-trial baselines hold few spikes) and gives a different answer. Here we
use the 10 ms rebinned counts and no smoothing. Pass `hamming_ms=40` for the paper's
version, which also smears the onset slightly earlier.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, exp_type in zip(axs, ["single_E", "single_I"]):
    for dur in np.unique(d["cond_dur1_ms"][d["cond_exp_type"] == exp_type]):
        p = ta.psth(c10[ta.select(d, exp_type=exp_type, dur1_ms=dur, contrast=0)], t10, 0.01)
        ax.plot(t10, p[0], label=f"E, dur {dur:g} ms")
        ax.plot(t10, p[1], "--", label=f"I, dur {dur:g} ms")
    ax.axvline(0, color="k", lw=0.5); ax.set_xlim(-0.2, 0.6)
    ax.set_title(f"{exp_type}, laser only"); ax.set_xlabel("time from onset (s)")
axs[0].set_ylabel("rate / baseline"); axs[1].legend(fontsize=7)
fig.tight_layout()

Raw mean rates in Hz, with the trial-to-trial spread (SEM):

In [ ]:
rate = c10[ta.select(d, exp_type="single_E", dur1_ms=1.0, contrast=0)] / 0.01   # (n, 2, n_bins) Hz
mean, sem = rate.mean(0), rate.std(0, ddof=1) / np.sqrt(len(rate))
plt.figure(figsize=(6, 3))
for pop, name in enumerate("EI"):
    plt.plot(t10, mean[pop], label=name)
    plt.fill_between(t10, mean[pop] - sem[pop], mean[pop] + sem[pop], alpha=0.3)
plt.xlim(-0.2, 0.6); plt.xlabel("time from onset (s)"); plt.ylabel("Hz"); plt.legend();

## 6. Several mice, and cross-validation folds

Every file has the same keys, so pooling is a loop. Condition sets differ per mouse
(e.g. M151020 sweeps more durations), so match conditions by their columns, not by row
index. Counts below are laser-only paired_EE trials.

In [ ]:
for f in sorted(ta.RESULTS.glob("trial_counts_b30_*_s1.npz")):
    di = dict(np.load(f))
    m = ta.select(di, exp_type="paired_EE", contrast=0)
    print(f"{str(di['animal_id']):15s} trials={len(di['counts']):5d}  paired_EE={m.sum():4d}  "
          f"ipis={np.unique(di['cond_ipi_ms'][di['cond_exp_type'] == 'paired_EE']).tolist()}")

Folds are a downstream choice now. `ta.fold_labels` makes a seeded k-fold split within
each condition.

In [ ]:
fold = ta.fold_labels(d["cond_idx"])
m = ta.select(d, exp_type="single_E", dur1_ms=1.0, contrast=0)
train = ta.psth(c10[m & (fold != 0)], t10, 0.01)
test  = ta.psth(c10[m & (fold == 0)], t10, 0.01)
print("train/test E PSTH correlation:", np.corrcoef(train[0], test[0])[0, 1].round(3))